# Notebook 03 — MO Codes Engineering

## Objective
This notebook transforms the raw `mo_codes` field from the cleaned collision dataset into a structured analytical MO layer.

## Why this notebook is important
The raw `mo_codes` field is one of the richest contextual fields in the traffic collision dataset, but in its original form it is not suitable for direct analytical use because:

- one collision can contain multiple MO codes in the same cell
- raw numeric codes are not business-readable on their own
- the field must be normalized before it can support grouping, filtering, modeling, or dashboarding

## Main goals of this notebook
In this notebook, we will:

- load the cleaned collision base from Notebook 02
- load the classified MO reference file
- split multi-code MO values into individual codes
- create a collision-to-code bridge table
- join the bridge table with the classified MO reference
- identify unmapped MO codes
- build a clean `dim_mo_codes` table
- prepare a simple enriched analytical MO layer

## Analytical design choice
To keep the project practical and focused, the main analytical grouping in this notebook will rely primarily on:

- `Analytical_Category`
- `Analytical_Subcategory`

Additional classification fields may be preserved in the reference layer, but the main analysis will stay centered on these two levels for clarity and usability.

In [1]:
# ============================================================
# Notebook 03 — Environment Setup and File Validation
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Project path resolution
# ------------------------------------------------------------
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"

# Create notebook-specific output folders
MO_TABLES_DIR = TABLES_DIR / "mo_analysis"
MO_FIGURES_DIR = FIGURES_DIR / "mo_analysis"

MO_TABLES_DIR.mkdir(parents=True, exist_ok=True)
MO_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Input files
# ------------------------------------------------------------
COLLISIONS_CLEAN_FILE = CLEANED_DIR / "collisions_clean.csv"
MO_CLASSIFIED_FILE = RAW_DIR / "MO_Codes_Classified_for_Traffic_Project.csv"
MO_REFERENCE_FILE = RAW_DIR / "Mo Codes.xlsx"

# ------------------------------------------------------------
# File validation
# ------------------------------------------------------------
assert COLLISIONS_CLEAN_FILE.exists(), f"Missing cleaned collisions file: {COLLISIONS_CLEAN_FILE}"
assert MO_CLASSIFIED_FILE.exists(), f"Missing classified MO file: {MO_CLASSIFIED_FILE}"
assert MO_REFERENCE_FILE.exists(), f"Missing MO reference file: {MO_REFERENCE_FILE}"

print("=" * 70)
print("Notebook 03 — Environment Setup")
print("=" * 70)
print(f"Current directory : {CURRENT_DIR}")
print(f"Project root      : {PROJECT_ROOT}")
print(f"Raw data dir      : {RAW_DIR}")
print(f"Cleaned data dir  : {CLEANED_DIR}")
print(f"Tables dir        : {TABLES_DIR}")
print(f"Figures dir       : {FIGURES_DIR}")
print("-" * 70)
print("Expected input files:")
print(f"Clean collisions file       -> {COLLISIONS_CLEAN_FILE}")
print(f"MO classified reference     -> {MO_CLASSIFIED_FILE}")
print(f"MO original reference       -> {MO_REFERENCE_FILE}")
print("-" * 70)
print("Existence check:")
print(f"Clean collisions exists?    {COLLISIONS_CLEAN_FILE.exists()}")
print(f"MO classified exists?       {MO_CLASSIFIED_FILE.exists()}")
print(f"MO reference exists?        {MO_REFERENCE_FILE.exists()}")
print("=" * 70)

Notebook 03 — Environment Setup
Current directory : c:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\notebooks
Project root      : c:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project
Raw data dir      : c:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\data\raw
Cleaned data dir  : c:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\data\cleaned
Tables dir        : c:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tables
Figures dir       : c:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\figures
----------------------------------------------------------------------
Expected input files:
Clean collisions file       -> c:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\data\cleaned\collisions_clean.csv
MO classified reference     -> c:\Users\MKamal\Desktop\Project\without output\la_traffic_collis

## Step 1 — Load the Clean Collision Base and the MO Classification Reference

In this step, we load the two main inputs required for MO engineering:

- the cleaned collision base from Notebook 02
- the classified MO reference file prepared for this project

### Why this step matters
Notebook 03 depends on connecting collision records to the analytical MO classification layer.

This requires:
- a clean collision table that still contains the raw `mo_codes` field
- a classified MO reference that provides the analytical labels needed for grouping and interpretation

### Main objective of this step
Before splitting MO codes, we first need to confirm:
- the shape of both input files
- the available columns
- the schema of the classified MO file
- whether the classification file already contains the fields we want to rely on analytically

### Analytical design note
The main grouping logic in this notebook will focus primarily on:
- `Analytical_Category`
- `Analytical_Subcategory`

Other reference columns may still be preserved, but these two fields will be the main analytical levels used later.

In [2]:
# ============================================================
# Cell 2: Load cleaned collisions and classified MO reference
# ============================================================

# Load main inputs
collisions = pd.read_csv(COLLISIONS_CLEAN_FILE, low_memory=False)
mo_classified = pd.read_csv(MO_CLASSIFIED_FILE, low_memory=False)

print("=" * 70)
print("Main Input Files Loaded Successfully")
print("=" * 70)

print("Clean collisions dataset:")
print(f"Shape: {collisions.shape}")
print(f"Columns: {len(collisions.columns)}")
print("-" * 70)
print("First 10 columns:")
for i, col in enumerate(collisions.columns[:10], start=1):
    print(f"{i:02d}. {col}")

print("-" * 70)
print("MO classified reference dataset:")
print(f"Shape: {mo_classified.shape}")
print(f"Columns: {len(mo_classified.columns)}")
print("-" * 70)
print("MO classified columns:")
for i, col in enumerate(mo_classified.columns, start=1):
    print(f"{i:02d}. {col}")

print("-" * 70)
print("Clean collisions preview:")
display(collisions.head(5))

print("-" * 70)
print("MO classified preview:")
display(mo_classified.head(10))

print("=" * 70)

Main Input Files Loaded Successfully
Clean collisions dataset:
Shape: (621677, 34)
Columns: 34
----------------------------------------------------------------------
First 10 columns:
01. dr_number
02. date_reported
03. date_occurred
04. time_occurred
05. time_occurred_str
06. occ_hour
07. occ_minute
08. occ_time_of_day
09. occ_year
10. occ_month
----------------------------------------------------------------------
MO classified reference dataset:
Shape: (777, 21)
Columns: 21
----------------------------------------------------------------------
MO classified columns:
01. Code
02. Code_Number
03. Description
04. Code_Family
05. Analytical_Domain
06. Analytical_Category
07. Analytical_Subcategory
08. Primary_Entity
09. Dashboard_Group
10. Traffic_Relevance_Score
11. Traffic_Relevance_Band
12. Keep_in_Main_Collision_Dashboard
13. Modeling_Role
14. Recommended_Use
15. Flag_Traffic_Core
16. Flag_Vehicle_or_RoadUser
17. Flag_Person_Profile
18. Flag_Location_Infrastructure
19. Flag_Behavior

,dr_number,date_reported,date_occurred,time_occurred,time_occurred_str,occ_hour,occ_minute,occ_time_of_day,occ_year,occ_month,...,is_valid_victim_descent_analytical,premise_code,premise_description,has_complete_premise,address,cross_street,location,latitude,longitude,has_valid_coordinates
0,212013850,2021-09-03,2021-09-02,2335,2335,23,35,Night,2021,9,...,True,101.0,STREET,True,WILTON PL,6TH ST,"(34.063, -118.3141)",34.0630,-118.3141,True
1,221417787,2022-10-17,2022-10-17,1620,1620,16,20,Afternoon,2022,10,...,False,101.0,STREET,True,NATIONAL BL,MOTOR AV,"(34.029, -118.4113)",34.0290,-118.4113,True
2,221418141,2022-10-26,2022-10-26,1135,1135,11,35,Morning,2022,10,...,False,101.0,STREET,True,PALMS BL,ROSEWOOD AV,"(34.0052, -118.4478)",34.0052,-118.4478,True
3,222017859,2022-12-01,2022-12-01,230,230,2,30,Late Night,2022,12,...,True,101.0,STREET,True,IROLO ST,SAN MARINO ST,"(34.0545, -118.3009)",34.0545,-118.3009,True
4,190319651,2019-08-24,2019-08-24,450,450,4,50,Late Night,2019,8,...,True,101.0,STREET,True,JEFFERSON BL,NORMANDIE AV,"(34.0255, -118.3002)",34.0255,-118.3002,True


----------------------------------------------------------------------
MO classified preview:


,Code,Code_Number,Description,Code_Family,Analytical_Domain,Analytical_Category,Analytical_Subcategory,Primary_Entity,Dashboard_Group,Traffic_Relevance_Score,...,Keep_in_Main_Collision_Dashboard,Modeling_Role,Recommended_Use,Flag_Traffic_Core,Flag_Vehicle_or_RoadUser,Flag_Person_Profile,Flag_Location_Infrastructure,Flag_Behavior_Tactic,Flag_Severity_Harm,Flag_Jurisdiction_Admin
0,100,100,Suspect Impersonate,100s,Approach / Pretext,Approach Strategy,Identity / role impersonation,Suspect,Narrative enrichment,0,...,No,Exclude from main collision model,Exclude from the core collision mart and keep ...,0,0,1,0,1,0,0
1,101,101,Aid victim,100s,Approach / Pretext,Approach Strategy,Helpful / familiar-contact pretext,Suspect,Narrative enrichment,0,...,No,Exclude from main collision model,Exclude from the core collision mart and keep ...,0,0,1,0,1,0,0
2,102,102,Blind,100s,Approach / Pretext,Approach Strategy,Disability / sympathy pretext,Suspect,Narrative enrichment,0,...,No,Exclude from main collision model,Exclude from the core collision mart and keep ...,0,0,1,0,1,0,0
3,103,103,Crippled,100s,Approach / Pretext,Approach Strategy,Disability / sympathy pretext,Suspect,Narrative enrichment,0,...,No,Exclude from main collision model,Exclude from the core collision mart and keep ...,0,0,1,0,1,0,0
4,104,104,Customer,100s,Approach / Pretext,Approach Strategy,Identity / role impersonation,Suspect,Narrative enrichment,0,...,No,Exclude from main collision model,Exclude from the core collision mart and keep ...,0,0,1,0,1,0,0
5,105,105,Delivery,100s,Approach / Pretext,Approach Strategy,Authority / service-role pretext,Suspect,Narrative enrichment,0,...,No,Exclude from main collision model,Exclude from the core collision mart and keep ...,0,0,1,0,1,0,0
6,106,106,Doctor,100s,Approach / Pretext,Approach Strategy,Authority / service-role pretext,Suspect,Narrative enrichment,0,...,No,Exclude from main collision model,Exclude from the core collision mart and keep ...,0,0,1,0,1,0,0
7,107,107,God,100s,Approach / Pretext,Approach Strategy,Identity / role impersonation,Suspect,Narrative enrichment,0,...,No,Exclude from main collision model,Exclude from the core collision mart and keep ...,0,0,1,0,1,0,0
8,108,108,Infirm,100s,Approach / Pretext,Approach Strategy,Disability / sympathy pretext,Suspect,Narrative enrichment,0,...,No,Exclude from main collision model,Exclude from the core collision mart and keep ...,0,0,1,0,1,0,0
9,109,109,Inspector,100s,Approach / Pretext,Approach Strategy,Authority / service-role pretext,Suspect,Narrative enrichment,0,...,No,Exclude from main collision model,Exclude from the core collision mart and keep ...,0,0,1,0,1,0,0


## Step 2 — Standardize the MO Classification Schema

The classified MO file contains a rich analytical structure, but before using it in joins and downstream modeling, we need to standardize its schema and define the fields that will be used in this notebook.

### Why this step matters
Reliable MO engineering requires a stable reference table with:
- consistent column naming
- a standardized MO code key
- clearly defined analytical fields
- reduced schema ambiguity before joining

### Main objective
In this step, we will:
- rename the classification columns into a clean analytical format
- standardize the MO code field
- define the subset of reference columns needed for the project
- keep the notebook focused on the two main grouping levels:
  - `analytical_category`
  - `analytical_subcategory`

### Design choice
The classified file may contain additional fields such as:
- analytical domain
- dashboard group
- relevance score
- usage flags

These fields may still be preserved in the dimension table, but the main analytical logic of this notebook will remain centered on:
- category
- subcategory

In [3]:
# ============================================================
# Cell 3: Standardize MO classification schema
# ============================================================

# Preserve original classified reference
mo_classified_raw = mo_classified.copy()

# Rename columns to analytical snake_case format
mo_classified = mo_classified.rename(columns={
    "Code": "mo_code",
    "Code_Number": "mo_code_number",
    "Description": "mo_description",
    "Code_Family": "code_family",
    "Analytical_Domain": "analytical_domain",
    "Analytical_Category": "analytical_category",
    "Analytical_Subcategory": "analytical_subcategory",
    "Primary_Entity": "primary_entity",
    "Dashboard_Group": "dashboard_group",
    "Traffic_Relevance_Score": "traffic_relevance_score",
    "Traffic_Relevance_Band": "traffic_relevance_band",
    "Keep_in_Main_Collision_Dashboard": "keep_in_main_collision_dashboard",
    "Modeling_Role": "modeling_role",
    "Recommended_Use": "recommended_use",
    "Flag_Traffic_Core": "flag_traffic_core",
    "Flag_Vehicle_or_RoadUser": "flag_vehicle_or_road_user",
    "Flag_Person_Profile": "flag_person_profile",
    "Flag_Location_Infrastructure": "flag_location_infrastructure",
    "Flag_Behavior_Tactic": "flag_behavior_tactic",
    "Flag_Severity_Harm": "flag_severity_harm",
    "Flag_Jurisdiction_Admin": "flag_jurisdiction_admin"
})

# Standardize MO code as zero-free clean string key
mo_classified["mo_code"] = (
    pd.to_numeric(mo_classified["mo_code"], errors="coerce")
    .astype("Int64")
    .astype(str)
)

# Define the reference columns we plan to keep
mo_reference_columns = [
    "mo_code",
    "mo_code_number",
    "mo_description",
    "analytical_category",
    "analytical_subcategory",
    "analytical_domain",
    "code_family",
    "primary_entity",
    "dashboard_group",
    "traffic_relevance_score",
    "traffic_relevance_band",
    "keep_in_main_collision_dashboard",
    "modeling_role",
    "recommended_use"
]

mo_reference = mo_classified[mo_reference_columns].copy()

# Validation summary
mo_schema_summary = pd.DataFrame({
    "metric": [
        "row_count",
        "column_count_after_standardization",
        "non_null_mo_code_count",
        "distinct_mo_code_count",
        "duplicate_mo_code_count",
        "non_null_analytical_category_count",
        "non_null_analytical_subcategory_count"
    ],
    "value": [
        len(mo_reference),
        mo_reference.shape[1],
        mo_reference["mo_code"].notna().sum(),
        mo_reference["mo_code"].nunique(dropna=True),
        mo_reference.duplicated(subset=["mo_code"]).sum(),
        mo_reference["analytical_category"].notna().sum(),
        mo_reference["analytical_subcategory"].notna().sum()
    ]
})

print("=" * 70)
print("MO Classification Schema Standardization Completed")
print("=" * 70)
display(mo_schema_summary)

print("-" * 70)
print("Standardized MO reference columns:")
for i, col in enumerate(mo_reference.columns, start=1):
    print(f"{i:02d}. {col}")

print("-" * 70)
print("MO reference preview:")
display(mo_reference.head(10))

print("=" * 70)

MO Classification Schema Standardization Completed


,metric,value
0,row_count,777
1,column_count_after_standardization,14
2,non_null_mo_code_count,777
3,distinct_mo_code_count,777
4,duplicate_mo_code_count,0
5,non_null_analytical_category_count,777
6,non_null_analytical_subcategory_count,777


----------------------------------------------------------------------
Standardized MO reference columns:
01. mo_code
02. mo_code_number
03. mo_description
04. analytical_category
05. analytical_subcategory
06. analytical_domain
07. code_family
08. primary_entity
09. dashboard_group
10. traffic_relevance_score
11. traffic_relevance_band
12. keep_in_main_collision_dashboard
13. modeling_role
14. recommended_use
----------------------------------------------------------------------
MO reference preview:


,mo_code,mo_code_number,mo_description,analytical_category,analytical_subcategory,analytical_domain,code_family,primary_entity,dashboard_group,traffic_relevance_score,traffic_relevance_band,keep_in_main_collision_dashboard,modeling_role,recommended_use
0,100,100,Suspect Impersonate,Approach Strategy,Identity / role impersonation,Approach / Pretext,100s,Suspect,Narrative enrichment,0,0 - Exclude,No,Exclude from main collision model,Exclude from the core collision mart and keep ...
1,101,101,Aid victim,Approach Strategy,Helpful / familiar-contact pretext,Approach / Pretext,100s,Suspect,Narrative enrichment,0,0 - Exclude,No,Exclude from main collision model,Exclude from the core collision mart and keep ...
2,102,102,Blind,Approach Strategy,Disability / sympathy pretext,Approach / Pretext,100s,Suspect,Narrative enrichment,0,0 - Exclude,No,Exclude from main collision model,Exclude from the core collision mart and keep ...
3,103,103,Crippled,Approach Strategy,Disability / sympathy pretext,Approach / Pretext,100s,Suspect,Narrative enrichment,0,0 - Exclude,No,Exclude from main collision model,Exclude from the core collision mart and keep ...
4,104,104,Customer,Approach Strategy,Identity / role impersonation,Approach / Pretext,100s,Suspect,Narrative enrichment,0,0 - Exclude,No,Exclude from main collision model,Exclude from the core collision mart and keep ...
5,105,105,Delivery,Approach Strategy,Authority / service-role pretext,Approach / Pretext,100s,Suspect,Narrative enrichment,0,0 - Exclude,No,Exclude from main collision model,Exclude from the core collision mart and keep ...
6,106,106,Doctor,Approach Strategy,Authority / service-role pretext,Approach / Pretext,100s,Suspect,Narrative enrichment,0,0 - Exclude,No,Exclude from main collision model,Exclude from the core collision mart and keep ...
7,107,107,God,Approach Strategy,Identity / role impersonation,Approach / Pretext,100s,Suspect,Narrative enrichment,0,0 - Exclude,No,Exclude from main collision model,Exclude from the core collision mart and keep ...
8,108,108,Infirm,Approach Strategy,Disability / sympathy pretext,Approach / Pretext,100s,Suspect,Narrative enrichment,0,0 - Exclude,No,Exclude from main collision model,Exclude from the core collision mart and keep ...
9,109,109,Inspector,Approach Strategy,Authority / service-role pretext,Approach / Pretext,100s,Suspect,Narrative enrichment,0,0 - Exclude,No,Exclude from main collision model,Exclude from the core collision mart and keep ...


## Step 3 — Profile the Raw `mo_codes` Field Before Splitting

Before exploding MO codes into a bridge table, we first need to understand the structure of the raw `mo_codes` field inside the cleaned collision dataset.

### Why this step matters
The raw `mo_codes` field still contains multiple codes inside the same cell.
Before splitting it, we should document:

- how many rows contain MO codes
- how many rows are missing MO codes
- how many codes typically appear in one collision
- whether duplicate MO codes may appear within the same collision
- whether the raw formatting is stable enough for controlled splitting

### Main objective
This step creates a structural baseline for the MO explosion process.
It helps us validate that the upcoming split/explode logic is grounded in the real content of the field rather than assumptions.

### Methodological note
At this stage, we do not yet reshape the dataset.
We only profile the raw field and prepare the splitting logic based on actual observed patterns.

In [4]:
# ============================================================
# Cell 4: Profile raw mo_codes before split/explode
# ============================================================

# Preserve working copy
mo_work = collisions[["dr_number", "mo_codes"]].copy()

# Basic flags
mo_work["has_mo_codes"] = mo_work["mo_codes"].notna()

# Standardize raw string spacing for profiling only
mo_work["mo_codes_str"] = mo_work["mo_codes"].astype(str).str.strip()
mo_work.loc[~mo_work["has_mo_codes"], "mo_codes_str"] = np.nan

# Split into list using whitespace
mo_work["mo_code_list_raw"] = mo_work["mo_codes_str"].apply(
    lambda x: x.split() if pd.notna(x) else []
)

# Count raw codes per collision
mo_work["mo_code_count_raw"] = mo_work["mo_code_list_raw"].apply(len)

# Count unique codes per collision
mo_work["mo_code_count_unique"] = mo_work["mo_code_list_raw"].apply(lambda x: len(set(x)))

# Flag rows with duplicates inside the same collision
mo_work["has_duplicate_mo_within_collision"] = (
    mo_work["mo_code_count_raw"] > mo_work["mo_code_count_unique"]
)

# Summary
mo_profile_summary = pd.DataFrame({
    "metric": [
        "total_collision_rows",
        "rows_with_mo_codes",
        "rows_without_mo_codes",
        "share_with_mo_codes_percent",
        "total_raw_mo_code_entries",
        "total_unique_mo_code_entries_within_rows",
        "rows_with_duplicate_codes_inside_same_collision",
        "max_raw_code_count_in_one_collision",
        "avg_raw_code_count_per_non_null_collision",
        "median_raw_code_count_per_non_null_collision"
    ],
    "value": [
        len(mo_work),
        mo_work["has_mo_codes"].sum(),
        (~mo_work["has_mo_codes"]).sum(),
        round(mo_work["has_mo_codes"].mean() * 100, 2),
        mo_work["mo_code_count_raw"].sum(),
        mo_work["mo_code_count_unique"].sum(),
        mo_work["has_duplicate_mo_within_collision"].sum(),
        mo_work["mo_code_count_raw"].max(),
        round(mo_work.loc[mo_work["has_mo_codes"], "mo_code_count_raw"].mean(), 2),
        mo_work.loc[mo_work["has_mo_codes"], "mo_code_count_raw"].median()
    ]
})

print("=" * 70)
print("Raw MO Codes Profiling Summary")
print("=" * 70)
display(mo_profile_summary)

print("-" * 70)
print("Distribution of number of MO codes per collision:")
display(
    mo_work["mo_code_count_raw"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("mo_code_count_raw")
    .reset_index(name="collision_count")
    .head(20)
)

print("-" * 70)
print("Sample non-null MO code rows:")
display(
    mo_work.loc[mo_work["has_mo_codes"], ["dr_number", "mo_codes", "mo_code_list_raw", "mo_code_count_raw"]]
    .head(15)
)

print("-" * 70)
print("Rows with duplicate MO codes inside the same collision (if any):")
display(
    mo_work.loc[
        mo_work["has_duplicate_mo_within_collision"],
        ["dr_number", "mo_codes", "mo_code_list_raw", "mo_code_count_raw", "mo_code_count_unique"]
    ].head(20)
)

print("=" * 70)

Raw MO Codes Profiling Summary


,metric,value
0,total_collision_rows,621677.00
1,rows_with_mo_codes,534353.00
2,rows_without_mo_codes,87324.00
3,share_with_mo_codes_percent,85.95
4,total_raw_mo_code_entries,3523096.00
5,total_unique_mo_code_entries_within_rows,3523096.00
6,rows_with_duplicate_codes_inside_same_collision,0.00
7,max_raw_code_count_in_one_collision,10.00
8,avg_raw_code_count_per_non_null_collision,6.59
9,median_raw_code_count_per_non_null_collision,7.00


----------------------------------------------------------------------
Distribution of number of MO codes per collision:


,mo_code_count_raw,collision_count
0,0,87324
1,1,24470
2,2,5072
3,3,10179
4,4,23801
5,5,76553
6,6,98184
7,7,87068
8,8,117486
9,9,73892


----------------------------------------------------------------------
Sample non-null MO code rows:


,dr_number,mo_codes,mo_code_list_raw,mo_code_count_raw
0,212013850,3004 3027 3034 4027 3036 3101 3401 3701,"[3004, 3027, 3034, 4027, 3036, 3101, 3401, 3701]",8
1,221417787,4027 3011 3028 3034 3037 3101 3401 3701,"[4027, 3011, 3028, 3034, 3037, 3101, 3401, 3701]",8
2,221418141,4027 3011 3025 3034 3037 3101 3401 3701,"[4027, 3011, 3025, 3034, 3037, 3101, 3401, 3701]",8
3,222017859,3003 0913 3026 3035 3037 3101 3401 3701 4020,"[3003, 0913, 3026, 3035, 3037, 3101, 3401, 370...",9
4,190319651,3036 3004 3026 3101 4003,"[3036, 3004, 3026, 3101, 4003]",5
5,190319680,3037 3006 3028 3030 3039 3101 4003,"[3037, 3006, 3028, 3030, 3039, 3101, 4003]",7
6,190413769,3101 3401 3701 3006 3030,"[3101, 3401, 3701, 3006, 3030]",5
7,190127578,0605 3101 3401 3701 3011 3034,"[0605, 3101, 3401, 3701, 3011, 3034]",6
8,190319695,0605 4025 3037 3004 3025 3101,"[0605, 4025, 3037, 3004, 3025, 3101]",6
9,190411883,3101 3401 3701 3003 3025 3029,"[3101, 3401, 3701, 3003, 3025, 3029]",6


----------------------------------------------------------------------
Rows with duplicate MO codes inside the same collision (if any):


,dr_number,mo_codes,mo_code_list_raw,mo_code_count_raw,mo_code_count_unique


## Step 4 — Split and Explode `mo_codes` into a Collision-to-Code Bridge Table

After profiling the raw `mo_codes` field, we can now normalize it into a row-level bridge structure.

### Why this step matters
The original `mo_codes` column stores multiple codes inside a single collision row.
That format is not suitable for:

- grouping
- filtering
- SQL modeling
- dashboard slicing
- category-level analysis

### Transformation logic
In this step, we will:

- keep only collision rows that contain MO codes
- split the raw MO code string into individual code values
- explode the code list so that each collision-code relationship becomes one row
- create a normalized bridge table:
  - one collision
  - one MO code
  - one row

### Expected result
The output of this step will be the project’s normalized MO bridge layer:
`bridge_collision_mo`

This table will later be joined to the classified MO reference to create the enriched analytical MO layer.

### Methodological note
Because the earlier profiling confirmed that there are no duplicate MO codes within the same collision row, the explosion can be applied directly without extra de-duplication logic at this stage.

In [5]:
# ============================================================
# Cell 5: Split and explode mo_codes into bridge_collision_mo
# ============================================================

# Keep only rows with MO codes
bridge_collision_mo = mo_work.loc[
    mo_work["has_mo_codes"],
    ["dr_number", "mo_codes", "mo_code_list_raw", "mo_code_count_raw"]
].copy()

# Explode MO codes into row-level structure
bridge_collision_mo = bridge_collision_mo.explode("mo_code_list_raw").reset_index(drop=True)

# Rename and standardize final bridge code column
bridge_collision_mo = bridge_collision_mo.rename(columns={
    "mo_code_list_raw": "mo_code"
})

bridge_collision_mo["mo_code"] = (
    pd.to_numeric(bridge_collision_mo["mo_code"], errors="coerce")
    .astype("Int64")
    .astype(str)
)

# Add row-level bridge validation flag
bridge_collision_mo["has_valid_mo_code_format"] = bridge_collision_mo["mo_code"].notna()

# Validation summary
bridge_summary = pd.DataFrame({
    "metric": [
        "bridge_row_count",
        "distinct_collision_count",
        "distinct_mo_code_count_in_bridge",
        "duplicate_collision_mo_pairs",
        "null_mo_code_rows",
        "min_codes_per_collision_before_explode",
        "max_codes_per_collision_before_explode"
    ],
    "value": [
        len(bridge_collision_mo),
        bridge_collision_mo["dr_number"].nunique(),
        bridge_collision_mo["mo_code"].nunique(dropna=True),
        bridge_collision_mo.duplicated(subset=["dr_number", "mo_code"]).sum(),
        bridge_collision_mo["mo_code"].isna().sum(),
        mo_work.loc[mo_work["has_mo_codes"], "mo_code_count_raw"].min(),
        mo_work.loc[mo_work["has_mo_codes"], "mo_code_count_raw"].max()
    ]
})

print("=" * 70)
print("bridge_collision_mo Built Successfully")
print("=" * 70)
display(bridge_summary)

print("-" * 70)
print("bridge_collision_mo preview:")
display(bridge_collision_mo.head(20))

print("-" * 70)
print("Sample MO code frequencies in bridge:")
display(
    bridge_collision_mo["mo_code"]
    .value_counts(dropna=False)
    .head(20)
    .rename_axis("mo_code")
    .reset_index(name="count")
)

print("=" * 70)

bridge_collision_mo Built Successfully


,metric,value
0,bridge_row_count,3523096
1,distinct_collision_count,534353
2,distinct_mo_code_count_in_bridge,346
3,duplicate_collision_mo_pairs,0
4,null_mo_code_rows,0
5,min_codes_per_collision_before_explode,1
6,max_codes_per_collision_before_explode,10


----------------------------------------------------------------------
bridge_collision_mo preview:


,dr_number,mo_codes,mo_code,mo_code_count_raw,has_valid_mo_code_format
0,212013850,3004 3027 3034 4027 3036 3101 3401 3701,3004,8,True
1,212013850,3004 3027 3034 4027 3036 3101 3401 3701,3027,8,True
2,212013850,3004 3027 3034 4027 3036 3101 3401 3701,3034,8,True
3,212013850,3004 3027 3034 4027 3036 3101 3401 3701,4027,8,True
4,212013850,3004 3027 3034 4027 3036 3101 3401 3701,3036,8,True
5,212013850,3004 3027 3034 4027 3036 3101 3401 3701,3101,8,True
6,212013850,3004 3027 3034 4027 3036 3101 3401 3701,3401,8,True
7,212013850,3004 3027 3034 4027 3036 3101 3401 3701,3701,8,True
8,221417787,4027 3011 3028 3034 3037 3101 3401 3701,4027,8,True
9,221417787,4027 3011 3028 3034 3037 3101 3401 3701,3011,8,True


----------------------------------------------------------------------
Sample MO code frequencies in bridge:


,mo_code,count
0,3101,454441
1,3401,361373
2,3701,361237
3,3004,268791
4,3037,252159
5,3030,204190
6,3028,184098
7,3026,148649
8,3036,131369
9,3006,118445


## Step 5 — Join the Bridge Table with the Classified MO Reference and Measure Mapping Quality

Now that the normalized collision-to-code bridge table has been created, the next step is to connect it to the analytical MO classification reference.

### Why this step matters
The bridge table alone only tells us which collision is linked to which MO code.
It does not yet provide analytical meaning.

By joining the bridge with the classified MO reference, we can enrich each collision-code record with:
- code description
- analytical category
- analytical subcategory
- supporting reference attributes

### Main objective
This step will help us answer a critical quality question:

**How completely do the MO codes used in the collision dataset map to the classified MO reference?**

### What we will validate
We will measure:
- total mapped bridge rows
- total unmapped bridge rows
- mapped and unmapped distinct MO codes
- mapping coverage percentage
- sample unmapped codes if any exist

### Methodological note
This is one of the most important validation steps in Notebook 03 because the downstream analytical quality depends directly on the completeness of this mapping.

In [6]:
# ============================================================
# Cell 6: Join bridge with MO reference and measure mapping quality
# ============================================================

# Left join bridge to classified MO reference
bridge_enriched = bridge_collision_mo.merge(
    mo_reference,
    how="left",
    on="mo_code",
    indicator=True
)

# Mapping flags
bridge_enriched["is_mapped_to_reference"] = bridge_enriched["_merge"].eq("both")

# Distinct code-level mapping summary
bridge_code_mapping = (
    bridge_enriched.groupby("mo_code", dropna=False)
    .agg(
        bridge_row_count=("dr_number", "size"),
        is_mapped_to_reference=("is_mapped_to_reference", "max")
    )
    .reset_index()
)

# Unmapped codes table
mo_unmapped_codes = bridge_code_mapping.loc[
    ~bridge_code_mapping["is_mapped_to_reference"]
].copy()

# Overall mapping summary
mapping_quality_summary = pd.DataFrame({
    "metric": [
        "total_bridge_rows",
        "mapped_bridge_rows",
        "unmapped_bridge_rows",
        "mapped_bridge_share_percent",
        "distinct_bridge_codes",
        "mapped_distinct_codes",
        "unmapped_distinct_codes"
    ],
    "value": [
        len(bridge_enriched),
        bridge_enriched["is_mapped_to_reference"].sum(),
        (~bridge_enriched["is_mapped_to_reference"]).sum(),
        round(bridge_enriched["is_mapped_to_reference"].mean() * 100, 2),
        bridge_enriched["mo_code"].nunique(dropna=True),
        bridge_code_mapping["is_mapped_to_reference"].sum(),
        (~bridge_code_mapping["is_mapped_to_reference"]).sum()
    ]
})

print("=" * 70)
print("MO Mapping Quality Summary")
print("=" * 70)
display(mapping_quality_summary)

print("-" * 70)
print("Mapped bridge preview:")
display(
    bridge_enriched[
        [
            "dr_number",
            "mo_code",
            "mo_description",
            "analytical_category",
            "analytical_subcategory",
            "is_mapped_to_reference"
        ]
    ].head(20)
)

print("-" * 70)
print("Unmapped MO codes (if any):")
display(mo_unmapped_codes.head(50))

print("-" * 70)
print("Top mapped analytical categories:")
display(
    bridge_enriched.loc[bridge_enriched["is_mapped_to_reference"], "analytical_category"]
    .value_counts(dropna=False)
    .head(20)
    .rename_axis("analytical_category")
    .reset_index(name="count")
)

print("=" * 70)

MO Mapping Quality Summary


,metric,value
0,total_bridge_rows,3523096.00
1,mapped_bridge_rows,3521494.00
2,unmapped_bridge_rows,1602.00
3,mapped_bridge_share_percent,99.95
4,distinct_bridge_codes,346.00
5,mapped_distinct_codes,337.00
6,unmapped_distinct_codes,9.00


----------------------------------------------------------------------
Mapped bridge preview:


,dr_number,mo_code,mo_description,analytical_category,analytical_subcategory,is_mapped_to_reference
0,212013850,3004,T/C – Vehicle vs vehicle,Traffic Collision,Collision counterpart / mode pairing,True
1,212013850,3027,T/C – (K) Fatal injury,Traffic Collision,Injury severity,True
2,212013850,3034,T/C – City property involved: Yes,Traffic Collision,Scene context / property / intersection,True
3,212013850,4027,T/C – West Traffic Division (WTD),Reporting Division,LAPD division / traffic division,True
4,212013850,3036,T/C – At intersection: Yes,Traffic Collision,Scene context / property / intersection,True
5,212013850,3101,T/C – Primary collision factor (A) in narrative,Traffic Collision Cause,Primary collision factor family,True
6,212013850,3401,T/C – Type of collision,Collision Pattern,Type of collision,True
7,212013850,3701,T/C – Movement preceding collision,Pre-Collision Movement,Movement preceding collision,True
8,221417787,4027,T/C – West Traffic Division (WTD),Reporting Division,LAPD division / traffic division,True
9,221417787,3011,T/C – Vehicle vs fixed object,Traffic Collision,Collision counterpart / mode pairing,True


----------------------------------------------------------------------
Unmapped MO codes (if any):


,mo_code,bridge_row_count,is_mapped_to_reference
77,1505,1,False
78,1513,1,False
79,1528,1,False
132,2053,1,False
133,2055,1,False
174,3031,71,False
188,3063,116,False
189,3064,307,False
344,947,1103,False


----------------------------------------------------------------------
Top mapped analytical categories:


,analytical_category,count
0,Traffic Collision,1801303
1,Traffic Collision Cause,486978
2,Reporting Division,372037
3,Collision Pattern,361373
4,Pre-Collision Movement,361237
5,Incident Context,83906
6,Special Collision Factors,16717
7,Other / Narrative,12366
8,Vehicle-Centered Interaction,9464
9,Forensic Evidence,4989


## Step 6 — Build `dim_mo_codes` and the Enriched Collision-MO Analytical Layer

After validating mapping quality, we can now construct the two main MO outputs of this notebook.

### Main outputs of this step

#### 1) `dim_mo_codes`
A clean MO reference dimension containing one row per mapped MO code and its analytical labels.

#### 2) `fact_collision_mo_enriched`
An enriched row-level table where each collision-code relationship is already joined with its analytical classification.

### Why this step matters
These two tables together form the analytical MO backbone of the project:

- `dim_mo_codes` supports SQL and BI dimension modeling
- `fact_collision_mo_enriched` supports direct analysis in Python and downstream exports

### Design logic
At this stage:
- mapped codes will form the main usable analytical layer
- unmapped codes will be preserved separately for audit and future refinement
- the main analysis will continue to rely primarily on:
  - `analytical_category`
  - `analytical_subcategory`

### Methodological note
This step finalizes the usable MO layer without discarding mapping-quality transparency.
Unmapped codes are not lost; they are isolated into a separate quality-control output.

In [7]:
# ============================================================
# Cell 7: Build dim_mo_codes and fact_collision_mo_enriched
# ============================================================

# ------------------------------------------------------------
# Keep mapped rows only for the main analytical MO layer
# ------------------------------------------------------------
fact_collision_mo_enriched = bridge_enriched.loc[
    bridge_enriched["is_mapped_to_reference"]
].copy()

# Select final analytical columns for enriched fact
fact_collision_mo_enriched = fact_collision_mo_enriched[
    [
        "dr_number",
        "mo_code",
        "mo_description",
        "analytical_category",
        "analytical_subcategory",
        "analytical_domain",
        "code_family",
        "primary_entity",
        "dashboard_group",
        "traffic_relevance_score",
        "traffic_relevance_band",
        "keep_in_main_collision_dashboard",
        "modeling_role",
        "recommended_use"
    ]
].copy()

# ------------------------------------------------------------
# Build dim_mo_codes from mapped reference rows only
# ------------------------------------------------------------
dim_mo_codes = (
    fact_collision_mo_enriched[
        [
            "mo_code",
            "mo_description",
            "analytical_category",
            "analytical_subcategory",
            "analytical_domain",
            "code_family",
            "primary_entity",
            "dashboard_group",
            "traffic_relevance_score",
            "traffic_relevance_band",
            "keep_in_main_collision_dashboard",
            "modeling_role",
            "recommended_use"
        ]
    ]
    .drop_duplicates()
    .sort_values(["mo_code"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Build final unmapped audit table
# ------------------------------------------------------------
mo_unmapped_codes = (
    bridge_enriched.loc[
        ~bridge_enriched["is_mapped_to_reference"],
        ["mo_code", "dr_number"]
    ]
    .groupby("mo_code", dropna=False)
    .agg(
        unmapped_bridge_row_count=("dr_number", "size"),
        affected_collision_count=("dr_number", "nunique")
    )
    .reset_index()
    .sort_values(["unmapped_bridge_row_count", "mo_code"], ascending=[False, True])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Validation summaries
# ------------------------------------------------------------
dim_mo_validation = pd.DataFrame({
    "metric": [
        "rows_in_dim_mo_codes",
        "distinct_mo_code_count",
        "duplicate_mo_code_rows",
        "non_null_analytical_category_count",
        "non_null_analytical_subcategory_count"
    ],
    "value": [
        len(dim_mo_codes),
        dim_mo_codes["mo_code"].nunique(dropna=True),
        dim_mo_codes.duplicated(subset=["mo_code"]).sum(),
        dim_mo_codes["analytical_category"].notna().sum(),
        dim_mo_codes["analytical_subcategory"].notna().sum()
    ]
})

fact_mo_validation = pd.DataFrame({
    "metric": [
        "rows_in_fact_collision_mo_enriched",
        "distinct_collision_count",
        "distinct_mo_code_count",
        "duplicate_collision_mo_pairs"
    ],
    "value": [
        len(fact_collision_mo_enriched),
        fact_collision_mo_enriched["dr_number"].nunique(),
        fact_collision_mo_enriched["mo_code"].nunique(dropna=True),
        fact_collision_mo_enriched.duplicated(subset=["dr_number", "mo_code"]).sum()
    ]
})

print("=" * 70)
print("dim_mo_codes Validation")
print("=" * 70)
display(dim_mo_validation)

print("-" * 70)
print("fact_collision_mo_enriched Validation")
print("-" * 70)
display(fact_mo_validation)

print("-" * 70)
print("dim_mo_codes preview:")
display(dim_mo_codes.head(20))

print("-" * 70)
print("fact_collision_mo_enriched preview:")
display(fact_collision_mo_enriched.head(20))

print("-" * 70)
print("Final mo_unmapped_codes table:")
display(mo_unmapped_codes)

print("=" * 70)

dim_mo_codes Validation


,metric,value
0,rows_in_dim_mo_codes,337
1,distinct_mo_code_count,337
2,duplicate_mo_code_rows,0
3,non_null_analytical_category_count,337
4,non_null_analytical_subcategory_count,337


----------------------------------------------------------------------
fact_collision_mo_enriched Validation
----------------------------------------------------------------------


,metric,value
0,rows_in_fact_collision_mo_enriched,3521494
1,distinct_collision_count,534352
2,distinct_mo_code_count,337
3,duplicate_collision_mo_pairs,0


----------------------------------------------------------------------
dim_mo_codes preview:


,mo_code,mo_description,analytical_category,analytical_subcategory,analytical_domain,code_family,primary_entity,dashboard_group,traffic_relevance_score,traffic_relevance_band,keep_in_main_collision_dashboard,modeling_role,recommended_use
0,100,Suspect Impersonate,Approach Strategy,Identity / role impersonation,Approach / Pretext,100s,Suspect,Narrative enrichment,0.0,0 - Exclude,No,Exclude from main collision model,Exclude from the core collision mart and keep ...
1,1001,Aid for vehicle,Lure / Offer,Mobility / transport lure,Lure / Solicitation,1000s,Suspect,Traffic supporting context,3.0,3 - Useful enrichment,Optional,Context enrichment feature,"Use only when present to enrich narrative, edg..."
2,1004,Assistant,Lure / Offer,General solicitation lure,Lure / Solicitation,1000s,Suspect,Exclude from main dashboard,0.0,0 - Exclude,No,Exclude from main collision model,Exclude from the core collision mart and keep ...
3,101,Aid victim,Approach Strategy,Helpful / familiar-contact pretext,Approach / Pretext,100s,Suspect,Narrative enrichment,0.0,0 - Exclude,No,Exclude from main collision model,Exclude from the core collision mart and keep ...
4,1012,Find a job,Lure / Offer,Opportunity / service lure,Lure / Solicitation,1000s,Suspect,Exclude from main dashboard,0.0,0 - Exclude,No,Exclude from main collision model,Exclude from the core collision mart and keep ...
5,1013,Food,Lure / Offer,General solicitation lure,Lure / Solicitation,1000s,Suspect,Exclude from main dashboard,0.0,0 - Exclude,No,Exclude from main collision model,Exclude from the core collision mart and keep ...
6,1014,Game,Lure / Offer,General solicitation lure,Lure / Solicitation,1000s,Suspect,Exclude from main dashboard,0.0,0 - Exclude,No,Exclude from main collision model,Exclude from the core collision mart and keep ...
7,1017,Information,Lure / Offer,General solicitation lure,Lure / Solicitation,1000s,Suspect,Exclude from main dashboard,0.0,0 - Exclude,No,Exclude from main collision model,Exclude from the core collision mart and keep ...
8,102,Blind,Approach Strategy,Disability / sympathy pretext,Approach / Pretext,100s,Suspect,Narrative enrichment,0.0,0 - Exclude,No,Exclude from main collision model,Exclude from the core collision mart and keep ...
9,1020,Narcotics,Lure / Offer,General solicitation lure,Lure / Solicitation,1000s,Suspect,Exclude from main dashboard,0.0,0 - Exclude,No,Exclude from main collision model,Exclude from the core collision mart and keep ...


----------------------------------------------------------------------
fact_collision_mo_enriched preview:


,dr_number,mo_code,mo_description,analytical_category,analytical_subcategory,analytical_domain,code_family,primary_entity,dashboard_group,traffic_relevance_score,traffic_relevance_band,keep_in_main_collision_dashboard,modeling_role,recommended_use
0,212013850,3004,T/C – Vehicle vs vehicle,Traffic Collision,Collision counterpart / mode pairing,Traffic Collision Core,3000s,Collision,Core collision analytics,5.0,5 - Core,Yes,Core collision dimension,"Use directly in dashboards, drill-downs, and m..."
1,212013850,3027,T/C – (K) Fatal injury,Traffic Collision,Injury severity,Traffic Collision Core,3000s,Collision,Collision severity,5.0,5 - Core,Yes,Core collision dimension,"Use directly in dashboards, drill-downs, and m..."
2,212013850,3034,T/C – City property involved: Yes,Traffic Collision,Scene context / property / intersection,Traffic Collision Core,3000s,Collision,Scene context,5.0,5 - Core,Yes,Core collision dimension,"Use directly in dashboards, drill-downs, and m..."
3,212013850,4027,T/C – West Traffic Division (WTD),Reporting Division,LAPD division / traffic division,Traffic Collision Jurisdiction,4000s,Jurisdiction,Geography / reporting area,5.0,5 - Core,Yes,Core collision dimension,"Use directly in dashboards, drill-downs, and m..."
4,212013850,3036,T/C – At intersection: Yes,Traffic Collision,Scene context / property / intersection,Traffic Collision Core,3000s,Collision,Scene context,5.0,5 - Core,Yes,Core collision dimension,"Use directly in dashboards, drill-downs, and m..."
5,212013850,3101,T/C – Primary collision factor (A) in narrative,Traffic Collision Cause,Primary collision factor family,Traffic Collision Causation,3100s,Collision,Causation / PCF,5.0,5 - Core,Yes,Core collision dimension,"Use directly in dashboards, drill-downs, and m..."
6,212013850,3401,T/C – Type of collision,Collision Pattern,Type of collision,Traffic Collision Type,3400s,Collision,Collision pattern,5.0,5 - Core,Yes,Core collision dimension,"Use directly in dashboards, drill-downs, and m..."
7,212013850,3701,T/C – Movement preceding collision,Pre-Collision Movement,Movement preceding collision,Traffic Collision Pre-Collision Movement,3700s,Collision,Movement before impact,5.0,5 - Core,Yes,Core collision dimension,"Use directly in dashboards, drill-downs, and m..."
8,221417787,4027,T/C – West Traffic Division (WTD),Reporting Division,LAPD division / traffic division,Traffic Collision Jurisdiction,4000s,Jurisdiction,Geography / reporting area,5.0,5 - Core,Yes,Core collision dimension,"Use directly in dashboards, drill-downs, and m..."
9,221417787,3011,T/C – Vehicle vs fixed object,Traffic Collision,Collision counterpart / mode pairing,Traffic Collision Core,3000s,Collision,Core collision analytics,5.0,5 - Core,Yes,Core collision dimension,"Use directly in dashboards, drill-downs, and m..."


----------------------------------------------------------------------
Final mo_unmapped_codes table:


,mo_code,unmapped_bridge_row_count,affected_collision_count
0,947,1103,1103
1,3064,307,307
2,3063,116,116
3,3031,71,71
4,1505,1,1
5,1513,1,1
6,1528,1,1
7,2053,1,1
8,2055,1,1


## Step 7 — Save the Core MO Engineering Outputs

At this point, the main MO engineering workflow has been completed successfully.

### Outputs to be saved
This notebook will save the core MO analytical outputs:

- `bridge_collision_mo.csv`
- `dim_mo_codes.csv`
- `fact_collision_mo_enriched.csv`
- `mo_unmapped_codes.csv`
- `mo_mapping_quality_report.csv`

### Why this step matters
Saving these files creates the project’s normalized and classified MO layer, which will support:

- exploratory analysis in Notebook 04
- SQL bridge and dimension modeling
- dashboard grouping by category and subcategory
- downstream exports to Excel, Tableau, and Power BI

### Methodological note
The saved outputs reflect the validated analytical MO structure:
- raw multi-code fields were normalized
- collision-to-code relationships were preserved
- mapped codes were enriched with analytical labels
- unmapped codes were isolated transparently for QA follow-up

In [8]:
# ============================================================
# Cell 8: Save MO engineering outputs
# ============================================================

# ------------------------------------------------------------
# Build mapping quality report
# ------------------------------------------------------------
mo_mapping_quality_report = pd.DataFrame([
    {"component": "bridge_collision_mo", "metric": "row_count", "value": len(bridge_collision_mo)},
    {"component": "bridge_collision_mo", "metric": "distinct_collision_count", "value": bridge_collision_mo["dr_number"].nunique()},
    {"component": "bridge_collision_mo", "metric": "distinct_mo_code_count", "value": bridge_collision_mo["mo_code"].nunique(dropna=True)},
    {"component": "bridge_collision_mo", "metric": "duplicate_collision_mo_pairs", "value": bridge_collision_mo.duplicated(subset=["dr_number", "mo_code"]).sum()},

    {"component": "fact_collision_mo_enriched", "metric": "row_count", "value": len(fact_collision_mo_enriched)},
    {"component": "fact_collision_mo_enriched", "metric": "distinct_collision_count", "value": fact_collision_mo_enriched["dr_number"].nunique()},
    {"component": "fact_collision_mo_enriched", "metric": "distinct_mo_code_count", "value": fact_collision_mo_enriched["mo_code"].nunique(dropna=True)},

    {"component": "dim_mo_codes", "metric": "row_count", "value": len(dim_mo_codes)},
    {"component": "dim_mo_codes", "metric": "distinct_mo_code_count", "value": dim_mo_codes["mo_code"].nunique(dropna=True)},

    {"component": "mapping_quality", "metric": "mapped_bridge_rows", "value": bridge_enriched["is_mapped_to_reference"].sum()},
    {"component": "mapping_quality", "metric": "unmapped_bridge_rows", "value": (~bridge_enriched["is_mapped_to_reference"]).sum()},
    {"component": "mapping_quality", "metric": "mapped_bridge_share_percent", "value": round(bridge_enriched["is_mapped_to_reference"].mean() * 100, 2)},
    {"component": "mapping_quality", "metric": "mapped_distinct_codes", "value": bridge_code_mapping["is_mapped_to_reference"].sum()},
    {"component": "mapping_quality", "metric": "unmapped_distinct_codes", "value": (~bridge_code_mapping["is_mapped_to_reference"]).sum()},
])

# ------------------------------------------------------------
# Define output paths
# ------------------------------------------------------------
bridge_collision_mo_path = CLEANED_DIR / "bridge_collision_mo.csv"
dim_mo_codes_path = CLEANED_DIR / "dim_mo_codes.csv"
fact_collision_mo_enriched_path = CLEANED_DIR / "fact_collision_mo_enriched.csv"
mo_unmapped_codes_path = CLEANED_DIR / "mo_unmapped_codes.csv"
mo_mapping_quality_report_path = CLEANED_DIR / "mo_mapping_quality_report.csv"

# ------------------------------------------------------------
# Save files
# ------------------------------------------------------------
bridge_collision_mo.to_csv(bridge_collision_mo_path, index=False)
dim_mo_codes.to_csv(dim_mo_codes_path, index=False)
fact_collision_mo_enriched.to_csv(fact_collision_mo_enriched_path, index=False)
mo_unmapped_codes.to_csv(mo_unmapped_codes_path, index=False)
mo_mapping_quality_report.to_csv(mo_mapping_quality_report_path, index=False)

print("=" * 70)
print("Notebook 03 Outputs Saved Successfully")
print("=" * 70)
print(f"bridge_collision_mo.csv        -> {bridge_collision_mo_path}")
print(f"dim_mo_codes.csv               -> {dim_mo_codes_path}")
print(f"fact_collision_mo_enriched.csv -> {fact_collision_mo_enriched_path}")
print(f"mo_unmapped_codes.csv          -> {mo_unmapped_codes_path}")
print(f"mo_mapping_quality_report.csv  -> {mo_mapping_quality_report_path}")

print("-" * 70)
print("MO mapping quality report preview:")
display(mo_mapping_quality_report)

print("-" * 70)
print("Saved file existence check:")
print(f"bridge_collision_mo exists?        {bridge_collision_mo_path.exists()}")
print(f"dim_mo_codes exists?               {dim_mo_codes_path.exists()}")
print(f"fact_collision_mo_enriched exists? {fact_collision_mo_enriched_path.exists()}")
print(f"mo_unmapped_codes exists?          {mo_unmapped_codes_path.exists()}")
print(f"mo_mapping_quality_report exists?  {mo_mapping_quality_report_path.exists()}")

print("=" * 70)

Notebook 03 Outputs Saved Successfully
bridge_collision_mo.csv        -> c:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\data\cleaned\bridge_collision_mo.csv
dim_mo_codes.csv               -> c:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\data\cleaned\dim_mo_codes.csv
fact_collision_mo_enriched.csv -> c:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\data\cleaned\fact_collision_mo_enriched.csv
mo_unmapped_codes.csv          -> c:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\data\cleaned\mo_unmapped_codes.csv
mo_mapping_quality_report.csv  -> c:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\data\cleaned\mo_mapping_quality_report.csv
----------------------------------------------------------------------
MO mapping quality report preview:


,component,metric,value
0,bridge_collision_mo,row_count,3523096.00
1,bridge_collision_mo,distinct_collision_count,534353.00
2,bridge_collision_mo,distinct_mo_code_count,346.00
3,bridge_collision_mo,duplicate_collision_mo_pairs,0.00
4,fact_collision_mo_enriched,row_count,3521494.00
5,fact_collision_mo_enriched,distinct_collision_count,534352.00
6,fact_collision_mo_enriched,distinct_mo_code_count,337.00
7,dim_mo_codes,row_count,337.00
8,dim_mo_codes,distinct_mo_code_count,337.00
9,mapping_quality,mapped_bridge_rows,3521494.00


----------------------------------------------------------------------
Saved file existence check:
bridge_collision_mo exists?        True
dim_mo_codes exists?               True
fact_collision_mo_enriched exists? True
mo_unmapped_codes exists?          True
mo_mapping_quality_report exists?  True


# Notebook 03 Conclusion

This notebook successfully transformed the raw `mo_codes` field from the cleaned collision dataset into a normalized, classified, and analysis-ready MO structure.

## What was completed

### 1) MO classification reference standardization
The project-specific MO classification file was loaded and standardized successfully.

- raw reference rows loaded: **777**
- standardized reference columns retained: **14**
- distinct reference MO codes: **777**
- duplicate MO codes in reference: **0**
- non-null `analytical_category`: **777**
- non-null `analytical_subcategory`: **777**

This confirmed that the classification file was structurally ready for joining and downstream analytical use.

### 2) Raw `mo_codes` field profiling
The cleaned collision base was profiled before splitting the multi-code field.

Key findings:
- total collision rows: **621,677**
- rows with non-null `mo_codes`: **534,353**
- rows without `mo_codes`: **87,324**
- share of collisions with MO codes: **85.95%**
- total raw MO code entries across all collisions: **3,523,096**
- average MO codes per non-null collision: **6.59**
- median MO codes per non-null collision: **7**
- maximum MO codes in one collision: **10**
- rows with duplicate MO codes inside the same collision: **0**

This confirmed that the field could be split directly without additional within-row de-duplication logic.

### 3) Bridge table creation
The raw multi-code field was normalized into a row-level collision-to-code bridge table.

Output created:
- `bridge_collision_mo.csv`

Bridge validation results:
- bridge rows: **3,523,096**
- distinct collisions in bridge: **534,353**
- distinct MO codes used in the collision dataset: **346**
- duplicate collision-MO pairs: **0**
- null MO code rows after explosion: **0**

This established the normalized MO linkage layer needed for SQL modeling, enrichment, and analysis.

### 4) Mapping to the classified MO reference
The bridge table was joined to the analytical MO reference to measure mapping completeness and enrich code meaning.

Mapping quality results:
- total bridge rows: **3,523,096**
- mapped bridge rows: **3,521,494**
- unmapped bridge rows: **1,602**
- mapped bridge share: **99.95%**
- distinct bridge MO codes: **346**
- mapped distinct codes: **337**
- unmapped distinct codes: **9**

This result confirms that the classified MO layer provides extremely strong analytical coverage of the collision dataset.

### 5) Unmapped MO code identification
A dedicated unmapped-code audit table was created.

Output created:
- `mo_unmapped_codes.csv`

Unmapped MO codes identified:
- `947`
- `1505`
- `1513`
- `1528`
- `2053`
- `2055`
- `3031`
- `3063`
- `3064`

Most impactful unmapped codes by bridge row count:
- `947` → **1,103**
- `3064` → **307**
- `3063` → **116**
- `3031` → **71**

The remaining unmapped codes appeared only once each.

### 6) MO dimension creation
A clean mapped MO dimension table was built successfully.

Output created:
- `dim_mo_codes.csv`

Validation results:
- rows in `dim_mo_codes`: **337**
- distinct MO codes: **337**
- duplicate MO code rows: **0**
- non-null `analytical_category`: **337**
- non-null `analytical_subcategory`: **337**

This table now serves as the reusable MO analytical reference dimension for the project.

### 7) Enriched row-level MO analytical layer
A mapped and enriched collision-code analytical table was created successfully.

Output created:
- `fact_collision_mo_enriched.csv`

Validation results:
- enriched fact rows: **3,521,494**
- distinct collisions represented: **534,352**
- distinct MO codes represented: **337**
- duplicate collision-MO pairs: **0**

This table now contains row-level collision-to-code relationships already enriched with:
- code description
- analytical category
- analytical subcategory
- analytical domain
- code family
- usage and relevance fields

### 8) Final saved outputs
The notebook saved the following files successfully:

- `bridge_collision_mo.csv`
- `dim_mo_codes.csv`
- `fact_collision_mo_enriched.csv`
- `mo_unmapped_codes.csv`
- `mo_mapping_quality_report.csv`

All files were confirmed to exist after export.

## Main analytical design result
To keep the MO layer practical and usable, the notebook centered its main analytical grouping on:

- `analytical_category`
- `analytical_subcategory`

This provided a balanced structure that is:
- simple enough for dashboards and reporting
- detailed enough for meaningful traffic interpretation
- suitable for Python, SQL, Excel, Tableau, and Power BI use

## Methodological result
This notebook established the project’s normalized and classified MO analytical layer.

The project now has:
- a clean collision-to-code bridge
- a reusable MO dimension
- a mapped enriched analytical MO fact layer
- a transparent unmapped-code audit output

## What comes next
The dataset is now ready for the next stage:

**Notebook 04 — EDA Core Analysis**

That notebook will focus on extracting the first major insights from the project, including:

- collision trends over time
- collisions by area and district
- premise patterns
- victim demographic patterns
- MO category and subcategory distributions
- severity and incident-context exploration
- the foundation for dashboard storytelling